In [ ]:
# Patch for matplotlib_inline 0.2.2 incompatibility with newer matplotlib.
# rcParams._get was removed; this restores it before ipykernel sets up the
# inline backend (which fires on kernel startup, before any other import).
import matplotlib
if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__

# keras-hexagdly → hls4ml export

This notebook shows the full path from training a hex CNN to exporting it for FPGA synthesis via [hls4ml](https://github.com/fastmachinelearning/hls4ml).

**The key idea:**  
keras-hexagdly uses a zig-zag grid representation internally — great for training on the CPU/GPU, but hls4ml cannot synthesize the HexBase sub-kernel decomposition.  
`patch_model_for_hls()` rewrites the trained model into an equivalent one built from *stock hls4ml-native layers* (`EinsumDense`, `MaxPooling1D`) that are numerically identical to the original, using the flat-pixel neighbor table derived from the same geometry.  
**The original model and its weights are never modified.**

```
Training / PC inference          hls4ml export
────────────────────────         ──────────────────────────────────
x (B, H, W, C)                   x (B, H, W, C)   ← same input
    │                                 │
    ▼                                 ▼
HexBase zig-zag conv      →   Reshape + EinsumDense   (gather folded in)
    │                                 │
    ▼                                 ▼
HexBase MaxPool           →   EinsumDense + MaxPooling1D(K,K)
    │                                 │
    ▼                                 ▼
y (same values, same shape)      y (bit-identical float32)
```

In [ ]:
import numpy as np
import keras
import keras_hexagdly as hgly

# notebook utilities shared with the other notebooks
from toy_data import ToyDataset, toy_hex_image
from hexplot import plot_hextensor

## 1. Train a small hex CNN

Same toy setup as `keras_hexagdly_cnn_example.ipynb`: four hexagonal shapes, a two-block Conv→Pool CNN.  
Nothing here is specific to the export — this is just a normal keras-hexagdly training run.

In [ ]:
shape_list = ['snowflake_2', 'snowflake_3', 'snowflake_4', 'double_hex']
H, W = 20, 20

train_data = ToyDataset(shape_list, 128, H, W).create(seed=0)
val_data   = ToyDataset(shape_list,  32, H, W).create(seed=1)
x_train, y_train = train_data.to_arrays()
x_val,   y_val   = val_data.to_arrays()
print('train:', x_train.shape, ' val:', x_val.shape)

In [ ]:
inp = keras.Input(shape=(H, W, 1), name='image')
x = hgly.Conv2d(out_channels=8, kernel_size=1, stride=1,
                share_neighbors=True, bias=False, name='conv1')(inp)
x = keras.layers.ReLU(name='relu1')(x)
x = hgly.MaxPool2d(kernel_size=1, stride=2, name='pool1')(x)
x = keras.layers.Flatten(name='flat')(x)
x = keras.layers.Dense(32, activation='relu', name='fc1')(x)
out = keras.layers.Dense(len(shape_list), name='logits')(x)
model = keras.Model(inp, out, name='hex_cnn')

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30, verbose=0,
)
print(f'final val accuracy: {history.history["val_accuracy"][-1]:.3f}')

## 2. Verify the trained model

Quick sanity check before we export: the model should classify a test image correctly.

In [ ]:
test_shape = 'snowflake_3'
testimage = toy_hex_image(test_shape, H, W, px=10, py=10)
plot_hextensor(testimage, figname='test_shape')

logits = model(keras.ops.convert_to_tensor(testimage))
probs  = keras.ops.convert_to_numpy(keras.ops.softmax(logits))[0]
order  = np.argsort(-probs)
for rank, idx in enumerate(order):
    tag = ' <-- correct' if shape_list[idx] == test_shape and rank == 0 else ''
    print(f'{rank+1}. {shape_list[idx]:>14s}  {100*probs[idx]:5.1f}%{tag}')

## 3. Patch the model for hls4ml

`patch_model_for_hls` returns a **new Keras model** — the original is untouched.

Internally it:
1. Sweeps single-pixel impulses through each hex layer's existing `call()` to measure the `(N_out, K)` neighbor table (no coordinate assumptions — exact by construction).
2. Folds the gather into an `EinsumDense` weight matrix for conv layers; uses `EinsumDense + MaxPooling1D` for pool layers.
3. Copies the trained weights into the replacement layers.
4. Passes all non-hex layers through unchanged.

No hls4ml dependency is needed yet — this step is pure Keras.

In [ ]:
from keras_hexagdly.hls4ml_ext import patch_model_for_hls
from keras_hexagdly.hls4ml_handler import register_hex_gather_layers

# strategy='gather': HexGather + HexRingMAC — sparse index table instead of
# dense selection matrices. Requires register_hex_gather_layers() before
# converting. Scales to full camera size.
#
# strategy='slotwise': K separate EinsumDense gather+MAC pairs (no custom handler).
# strategy='folded': one large EinsumDense kernel (C-sim only).
strategy = 'gather'

if strategy == 'gather':
    register_hex_gather_layers()

hls_model_input = patch_model_for_hls(model, strategy=strategy)
hls_model_input.summary()


## 4. Verify bit-identical outputs

The patched model must produce **exactly the same float32 output** as the original on the same input.  
This is the correctness guarantee: if this passes, the FPGA output will match PC inference up to fixed-point quantization.

In [ ]:
x_check = x_val[:8]  # a small batch

y_orig    = model.predict(x_check, verbose=0)
y_patched = hls_model_input.predict(x_check, verbose=0)

max_err = float(np.max(np.abs(y_orig - y_patched)))
print(f'max abs diff (original vs patched): {max_err:.2e}')
assert max_err < 1e-3, 'Outputs differ — something went wrong in patching!'
print('✓ outputs are numerically identical')

## 5. Convert to hls4ml and run C-simulation

The patched model uses **`strategy='gather'`** — `HexGather` + `HexRingMAC` layers
replace the hex layers with a sparse index-table lookup + ring-sharing MAC.

| Strategy | How it works | Synthesis-safe? |
|---|---|---|
| `gather` | Sparse index table → ROM lookup | Yes, any camera size |
| `slotwise` | K dense 0/1 matrices per slot | Only small grids (<9×9) |
| `folded` | One large dense matrix | C-sim only |


In [ ]:
import hls4ml

cfg = hls4ml.utils.config_from_keras_model(
    hls_model_input, granularity='name', backend='Vivado'
)

# smaller precision for the convolutional layer to reduce resource usage.
# cfg['conv1']['Precision'] = 'ap_fixed<16,6>'


### hls4ml implementation strategy

Two knobs control the hardware implementation:

- **`Strategy`**: `'Latency'` (default) fully unrolls the computation — one result per clock, maximum throughput, maximum LUT/DSP use. `'Resource'` time-multiplexes via `ReuseFactor` — fewer DSPs but more clock cycles per inference.
- **`ReuseFactor`**: integer ≥ 1. `1` = fully unrolled. `n` = same hardware reused `n` times → DSPs ÷ n, latency × n. Must divide the layer's total MAC count evenly.

Set globally on `cfg['Model']`, or per layer on `cfg['LayerName']['<name>']`.
In the slotwise export, the `_mac_k{i}` layers do the learned MACs (small `Cin×Cout`); the `_gather_k{i}` layers are ROM lookups where `ReuseFactor` has less effect.


In [ ]:
# ── Global settings (apply to all layers) ───────────────────────────────────

cfg['Model']['Precision']   = 'ap_fixed<32,12>'  # wide enough for C-sim accuracy
cfg['Model']['Strategy']    = 'Latency'           # 'Latency' or 'Resource'
cfg['Model']['ReuseFactor'] = 3                   # 1 = fully unrolled

# ── Per-layer override (optional) ────────────────────────────────────────────
# After patching, each hex layer is split into sublayers.
# Naming convention for slotwise Conv2d named 'conv1' with K neighbor slots:
#   conv1_gather_k{i}  — binary selection ROM
#   conv1_mac_k{i}     — learned MAC (Cin x Cout); target this for precision/reuse
#   conv1_add{i}       — pairwise Add
#
# Example: narrower precision on all MAC layers of conv1:
# for name in cfg['LayerName']:
#     if name.startswith('conv1_mac'):
#         cfg['LayerName'][name]['Precision'] = 'ap_fixed<8,4>'
#
# Example: Resource strategy + reuse on the Dense classifier:
# cfg['LayerName']['logits']['Strategy']    = 'Resource'
# cfg['LayerName']['logits']['ReuseFactor'] = 4

print('Available layer names:')
print(list(cfg['LayerName'].keys()))


In [ ]:
import os, shutil
if os.path.exists('/tmp/hls_hex_cnn'): shutil.rmtree('/tmp/hls_hex_cnn')

hls_model = hls4ml.converters.convert_from_keras_model(
    hls_model_input,
    hls_config=cfg,
    backend='Vivado',
    output_dir='/tmp/hls_hex_cnn',
    part='xcvu9p-flga2104-2L-e',
)
print('hls4ml model created — no custom layer needed')

In [ ]:
# Compile the HLS project with g++ (no Vivado needed for C-sim).
hls_model.compile()
print('C-simulation compiled successfully')

In [ ]:
y_csim = hls_model.predict(np.ascontiguousarray(x_check))

csim_err = float(np.max(np.abs(y_csim - y_orig)))
print(f'max abs diff (original vs C-sim): {csim_err:.4f}')
print('(residual is fixed-point quantization, not a logic error)')

# Classification accuracy should be preserved.
pred_orig = np.argmax(y_orig, axis=1)
pred_csim = np.argmax(y_csim, axis=1)
agreement = np.mean(pred_orig == pred_csim)
print(f'class-label agreement: {100*agreement:.1f}%')

In [ ]:
import os, stat, shutil

# Add Vitis HLS 2024.2 to PATH.
VITIS_BIN = '/tools/Xilinx/Vitis_HLS/2024.2/bin'
os.environ['PATH'] = VITIS_BIN + ':' + os.environ['PATH']

# vivado_hls wrapper (created once).
shim = os.path.expanduser('~/.local/bin/vivado_hls')
if not os.path.isfile(shim) or os.path.islink(shim):
    if os.path.islink(shim): os.remove(shim)
    os.makedirs(os.path.dirname(shim), exist_ok=True)
    with open(shim, 'w') as _f:
        _f.write('#!/bin/bash\nexec /tools/Xilinx/Vitis_HLS/2024.2/bin/vitis_hls "$@"\n')
    os.chmod(shim, os.stat(shim).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# NOTE: MaxPool2d still uses EinsumDense for gathering (strategy-agnostic).
# At 20x20 the pool gather matrix is 280,000 entries — same clang segfault
# as the old slotwise Conv2d issue.  MaxPool needs a HexGather equivalent
# (planned: Phase 5, nnet_hex_pool.h).  Until then, the synthesis model
# uses Conv2d only to demonstrate the gather improvement vs slotwise.
inp_s = keras.Input(shape=(H, W, 1), name='image')
xs = hgly.Conv2d(out_channels=8, kernel_size=1, stride=1,
                 share_neighbors=True, bias=False, name='conv1')(inp_s)
xs = keras.layers.ReLU(name='relu1')(xs)
xs = keras.layers.Flatten(name='flat')(xs)
xs = keras.layers.Dense(len(shape_list), name='logits')(xs)
synth_model = keras.Model(inp_s, xs, name='hex_cnn_synth')

synth_strategy = 'gather'
if synth_strategy == 'gather':
    register_hex_gather_layers()
synth_patched = patch_model_for_hls(synth_model, strategy=synth_strategy)
print(f'Synth model: {synth_patched.count_params():,} params (Conv2d only, no MaxPool)')


In [ ]:
# ── Synthesis config (edit before running the next cell) ────────────────────
cfg_s = hls4ml.utils.config_from_keras_model(synth_patched, granularity='name', backend='Vivado')
cfg_s['Model']['Precision']   = 'ap_fixed<16,6>'
cfg_s['Model']['Strategy']    = 'Latency'
cfg_s['Model']['ReuseFactor'] = 3

# Per-layer example:
# cfg_s['LayerName']['logits']['ReuseFactor'] = 4

print('Available layer names:')
print(list(cfg_s['LayerName'].keys()))


In [ ]:
# Allow Vitis HLS to use all available threads (helps post-clang phases).
os.environ['XILINX_HLS_JOBS'] = '8'

synth_dir = '/tmp/hls_hex_cnn_synth'
if os.path.exists(synth_dir): shutil.rmtree(synth_dir)

hls_synth = hls4ml.converters.convert_from_keras_model(
    synth_patched, hls_config=cfg_s, backend='Vivado',
    output_dir=synth_dir, part='xcvu9p-flga2104-2L-e',
)
hls_synth.build(csim=False, synth=True, export=False)
hls4ml.report.read_vivado_report(synth_dir)


## Summary

| Step | Code | What happens |
|---|---|---|
| Train | `model.fit(...)` | Normal keras-hexagdly training, zig-zag grid |
| Patch | `patch_model_for_hls(model, strategy='gather')` | Hex layers → HexGather + HexRingMAC |
| Register | `register_hex_gather_layers()` | Wires custom hls4ml handler + HLS templates |
| Verify | `np.max(abs(y_orig - y_patched)) < 1e-3` | Float32 bit-identical check |
| Convert | `hls4ml.converters.convert_from_keras_model(...)` | No custom HLS layer written by user |
| C-sim | `hls_model.compile(); hls_model.predict(x)` | HLS C++ correctness check |
| Synthesis | `hls_model.build(synth=True)` | Real FPGA resource / timing numbers |

The original `model` is never modified — you can continue training or saving it exactly as before.
